In [1]:
import pandas as pd

# Cargar el archivo CSV en un DataFrame de pandas
df = pd.read_csv('/content/autos.csv.csv')

# Mostrar las primeras 5 filas del DataFrame para una vista previa
# Muestra las primeras filas del DataFrame
display(df.head())

,ID,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Segmentation
0,462809,Male,No,22,No,Healthcare,1.0,Low,4.0,Cat_4,D
1,462643,Female,Yes,38,Yes,Engineer,NaN,Average,3.0,Cat_4,A
2,466315,Female,Yes,67,Yes,Engineer,1.0,Low,1.0,Cat_6,B
3,461735,Male,Yes,67,Yes,Lawyer,0.0,High,2.0,Cat_6,B
4,462669,Female,Yes,40,Yes,Entertainment,NaN,High,6.0,Cat_6,A


In [8]:
df.shape

(8068, 11)

### Validación de la calidad de los datos
Vamos a revisar la cantidad de valores nulos por columna para entender la integridad de nuestros datos.

In [9]:
# Calcular el número de valores nulos por columna
# Cuenta los nulos por columna
null_counts = df.isnull().sum()

# Calcular el porcentaje de valores nulos por columna
# Calcula el porcentaje de nulos
null_percentage = (df.isnull().sum() / len(df)) * 100


In [10]:
# Unir las Series de conteo de nulos y porcentaje de nulos en un DataFrame
# Unir conteos y porcentajes de nulos
null_summary = pd.DataFrame({
    'Conteo de Nulos': null_counts,
    'Porcentaje de Nulos': null_percentage
})

# Formatear la columna de porcentaje para mostrar el símbolo '%' y limitar a dos decimales
# Formatear porcentaje con '%'
null_summary['Porcentaje de Nulos'] = null_summary['Porcentaje de Nulos'].map(lambda x: f"{x:.2f}%")

# Mostrar la tabla combinada
# Mostrar tabla de resumen de nulos
display(null_summary)

,Conteo de Nulos,Porcentaje de Nulos
ID,0,0.00%
Gender,0,0.00%
Ever_Married,140,1.74%
Age,0,0.00%
Graduated,78,0.97%
Profession,124,1.54%
Work_Experience,829,10.28%
Spending_Score,0,0.00%
Family_Size,335,4.15%
Var_1,76,0.94%


In [13]:
# Identificar columnas con nulos para imputación
# Columnas a imputar
columns_to_impute = null_counts[null_counts > 0].index.tolist()

print("Columnas a imputar:", columns_to_impute)

# --- Estrategias de imputación ---

# Imputación para columnas categóricas (usando la moda)
# Columnas categóricas a imputar por moda
categorical_cols = ['Ever_Married', 'Graduated', 'Profession', 'Var_1']
for col in categorical_cols:
    if col in columns_to_impute:
        mode_value = df[col].mode()[0] # Calcular la moda
        df[col] = df[col].fillna(mode_value) # Rellenar nulos con la moda (sin inplace)
        print(f"Columna '{col}' imputada con la moda: {mode_value}")

# Imputación para columnas numéricas (usando la mediana)
# Columnas numéricas a imputar por mediana
numeric_cols = ['Work_Experience', 'Family_Size']
for col in numeric_cols:
    if col in columns_to_impute:
        median_value = df[col].median() # Calcular la mediana
        df[col] = df[col].fillna(median_value) # Rellenar nulos con la mediana (sin inplace)
        print(f"Columna '{col}' imputada con la mediana: {median_value}")

# Verificar de nuevo los nulos después de la imputación
# Verificar nulos después de imputación
print("\nValores nulos después de la imputación:")
display(df.isnull().sum())

Columnas a imputar: ['Ever_Married', 'Graduated', 'Profession', 'Work_Experience', 'Family_Size', 'Var_1']
Columna 'Ever_Married' imputada con la moda: Yes
Columna 'Graduated' imputada con la moda: Yes
Columna 'Profession' imputada con la moda: Artist
Columna 'Var_1' imputada con la moda: Cat_6
Columna 'Work_Experience' imputada con la mediana: 1.0
Columna 'Family_Size' imputada con la mediana: 3.0

Valores nulos después de la imputación:


,0
ID,0
Gender,0
Ever_Married,0
Age,0
Graduated,0
Profession,0
Work_Experience,0
Spending_Score,0
Family_Size,0
Var_1,0


### 1. Preprocesamiento: Codificación One-Hot para variables categóricas
Primero, convertiremos las variables categóricas a un formato numérico usando One-Hot Encoding.

In [15]:
from sklearn.preprocessing import OneHotEncoder

# Excluir la columna 'ID' y la variable objetivo 'Segmentation' de las características
# Excluir ID y Segmentation
X = df.drop(['ID', 'Segmentation'], axis=1)
y = df['Segmentation']

# Identificar columnas categóricas en X
# Columnas categóricas
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

# Inicializar OneHotEncoder
# Codificador One-Hot
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Aplicar One-Hot Encoding a las características categóricas
# Transformar categóricas
X_encoded_array = ohe.fit_transform(X[categorical_features])

# Obtener nombres de las nuevas columnas
# Nombres de nuevas columnas
encoded_feature_names = ohe.get_feature_names_out(categorical_features)

# Crear un DataFrame con las características codificadas
# DataFrame de características codificadas
X_encoded = pd.DataFrame(X_encoded_array, columns=encoded_feature_names, index=X.index)

# Eliminar las columnas categóricas originales y añadir las codificadas
# Unir DataFrame
X_preprocessed = pd.concat([
    X.drop(columns=categorical_features),
    X_encoded
], axis=1)

print("Dimensiones de X después del One-Hot Encoding:", X_preprocessed.shape)
display(X_preprocessed.head())

Dimensiones de X después del One-Hot Encoding: (8068, 28)


,Age,Work_Experience,Family_Size,Gender_Female,Gender_Male,Ever_Married_No,Ever_Married_Yes,Graduated_No,Graduated_Yes,Profession_Artist,...,Spending_Score_Average,Spending_Score_High,Spending_Score_Low,Var_1_Cat_1,Var_1_Cat_2,Var_1_Cat_3,Var_1_Cat_4,Var_1_Cat_5,Var_1_Cat_6,Var_1_Cat_7
0,22,1.0,4.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,38,1.0,3.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,67,1.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,67,0.0,2.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,40,1.0,6.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


### 2. División de datos en conjuntos de entrenamiento y prueba
Dividiremos el dataset preprocesado en conjuntos de entrenamiento y prueba para evaluar el rendimiento del modelo.

In [16]:
from sklearn.model_selection import train_test_split

# Dividir los datos en conjuntos de entrenamiento y prueba
# División de datos
X_train, X_test, y_train, y_test = train_test_split(X_preprocessed, y, test_size=0.2, random_state=42, stratify=y)

print("Dimensiones del conjunto de entrenamiento X_train:", X_train.shape)
print("Dimensiones del conjunto de prueba X_test:", X_test.shape)
print("Dimensiones del conjunto de entrenamiento y_train:", y_train.shape)
print("Dimensiones del conjunto de prueba y_test:", y_test.shape)

Dimensiones del conjunto de entrenamiento X_train: (6454, 28)
Dimensiones del conjunto de prueba X_test: (1614, 28)
Dimensiones del conjunto de entrenamiento y_train: (6454,)
Dimensiones del conjunto de prueba y_test: (1614,)


### 3. Escalado de características numéricas
Escalaremos las características numéricas para que tengan una media de 0 y una desviación estándar de 1, lo cual es crucial para el rendimiento de KNN.

In [17]:
from sklearn.preprocessing import StandardScaler

# Identificar columnas numéricas en el DataFrame preprocesado (las que no fueron One-Hot encoded)
# Columnas numéricas a escalar
numerical_features_to_scale = X_preprocessed.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Inicializar StandardScaler
# Escalador
scaler = StandardScaler()

# Aplicar escalado solo a las columnas numéricas en los conjuntos de entrenamiento y prueba
# Escalar X_train
X_train[numerical_features_to_scale] = scaler.fit_transform(X_train[numerical_features_to_scale])
# Escalar X_test
X_test[numerical_features_to_scale] = scaler.transform(X_test[numerical_features_to_scale])

print("Características numéricas escaladas en X_train (primeras filas):")
display(X_train[numerical_features_to_scale].head())

Características numéricas escaladas en X_train (primeras filas):


,Age,Work_Experience,Family_Size,Gender_Female,Gender_Male,Ever_Married_No,Ever_Married_Yes,Graduated_No,Graduated_Yes,Profession_Artist,...,Spending_Score_Average,Spending_Score_High,Spending_Score_Low,Var_1_Cat_1,Var_1_Cat_2,Var_1_Cat_3,Var_1_Cat_4,Var_1_Cat_5,Var_1_Cat_6,Var_1_Cat_7
917,-0.695320,1.970880,-1.231118,1.086126,-1.086126,1.213979,-1.213979,-0.766361,0.766361,1.434336,...,-0.566973,-0.424135,0.809241,-0.132891,-0.236549,-0.339781,-0.389985,-0.097682,0.717899,-0.159440
3398,1.703982,-0.456381,-0.564314,-0.920704,0.920704,-0.823738,0.823738,-0.766361,0.766361,-0.697187,...,1.763754,-0.424135,-1.235725,-0.132891,-0.236549,-0.339781,-0.389985,-0.097682,0.717899,-0.159440
2045,-0.635337,-0.456381,0.769294,1.086126,-1.086126,1.213979,-1.213979,-0.766361,0.766361,-0.697187,...,-0.566973,-0.424135,0.809241,-0.132891,-0.236549,-0.339781,-0.389985,-0.097682,0.717899,-0.159440
8060,0.264401,-0.759788,2.102901,1.086126,-1.086126,-0.823738,0.823738,-0.766361,0.766361,1.434336,...,1.763754,-0.424135,-1.235725,-0.132891,-0.236549,-0.339781,-0.389985,-0.097682,0.717899,-0.159440
4604,-0.935250,1.970880,-1.231118,1.086126,-1.086126,-0.823738,0.823738,1.304868,-1.304868,-0.697187,...,-0.566973,-0.424135,0.809241,-0.132891,-0.236549,-0.339781,-0.389985,-0.097682,-1.392953,6.271961


### 4. Entrenamiento y Evaluación del Modelo K-Nearest Neighbors (KNN)
Finalmente, entrenaremos el clasificador KNN y evaluaremos su precisión en el conjunto de prueba.

In [18]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Inicializar el clasificador KNN
# Clasificador KNN (con k=5 por defecto)
knn = KNeighborsClassifier(n_neighbors=5)

# Entrenar el modelo KNN con los datos escalados
# Entrenar modelo
print("\nEntrenando el modelo KNN...")
knn.fit(X_train, y_train)
print("Entrenamiento completado.")

# Realizar predicciones en el conjunto de prueba
# Predicciones
y_pred = knn.predict(X_test)

# Calcular la precisión del modelo
# Precisión del modelo
accuracy = accuracy_score(y_test, y_pred)
print(f"\nPrecisión del modelo KNN: {accuracy:.4f}")

# Pequeña interpretación de la precisión
# Interpretación
if accuracy > 0.8:
    print("El modelo KNN tiene una alta precisión, lo cual es prometedor.")
elif accuracy > 0.6:
    print("El modelo KNN tiene una precisión moderada, puede ser útil pero podría mejorarse.")
else:
    print("El modelo KNN tiene una baja precisión, se recomienda explorar otros modelos o técnicas de mejora.")


Entrenando el modelo KNN...
Entrenamiento completado.

Precisión del modelo KNN: 0.4771
El modelo KNN tiene una baja precisión, se recomienda explorar otros modelos o técnicas de mejora.
